# Scientific Data Processing

## Scientific Python for Engineers Workshop - Hour 2

**Author:** Dilip Kalagotla  
**Contact:** dilip.kalagotla@gmail.com

**Author:** Elijah LaLonde  
**Contact:** elalonde@fsu.edu

## Learning Objectives

- Load data from CSV files using Pandas
- Explore and clean datasets
- Perform numerical operations with NumPy
- Create publication-quality visualizations

---

## Part 1: Sample Experimental Data (Pandas Quick Intro)

In this first part of Hour 2, we will treat `Trajectory.csv` as a small experimental dataset and use it to practice loading, exploring, and visualizing data with Pandas, NumPy, and Matplotlib.

## Part 2: LES Cylinder CFD Data (Complex Visualization Example)

In the second part of this notebook, we will work with a subset of LES cylinder flow data using `lptlib`, and show how to load structured CFD results, make contour plots, and build simple animations of the flow field.

## Part 3: Spectral Proper Orthogonal Decompsition

In the third part of this notebook, we will generate sythetic data and perform modal analysis.

## Setup

First, let's import the libraries we'll need.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
import spod
import h5py
from matplotlib.animation import FuncAnimation
from matplotlib import cm
from IPython.display import HTML
import vtkplotlib as vpl
import ipywidgets as widgets
from ipywidgets import interactive
from scipy.signal import find_peaks

print("Libraries loaded successfully!")

---
## 1. Loading Data with Pandas

Pandas makes it easy to load data from various file formats:
- CSV: `pd.read_csv()`
- Excel: `pd.read_excel()`
- JSON: `pd.read_json()`
- And many more!

In [ ]:
# Load the sample data
df = pd.read_csv('data/Trajectory.csv')


---
## 2. Exploring the Data

In [ ]:
df.head()

In [ ]:
# Basic info about the DataFrame
# shape of the df -- one line -- shape()

# columns names in the df -- one line -- columns

In [ ]:
# Summary statistics
print("Summary statistics:")

# one line -- describe()

In [ ]:
# Data types
print("Data types:")
# one line -- dtypes

In [ ]:
# Check for missing values
print("Missing values per column:")

# one line -- isnull() sum()
print(df.isnull().sum())

---
## 3. Accessing Data

There are multiple ways to access data in a DataFrame.

In [ ]:
# Access a single column (returns a Series)
# one line -- remove 0 and type df['x'] or  df.loc[2]
x = 0
print(f"x column type: {type(x)}")
print(x.head())

In [ ]:
# Access multiple columns (returns a DataFrame)
# one line -- df[['x','y', 'z']]
position_data_data = 0
position_data_data.head()

In [ ]:
# Access by row index using .iloc (integer location)
print("Row 5:")
# one line --- .iloc[5]

In [ ]:
# Access by row and column
print(f"x position at row 5: {df.iloc[5]['x']}")
print(f"Or using loc: {df.loc[5, 'x']}")

In [ ]:
# Boolean indexing (filtering)
high_x = df[df['x'] > 12]
print(f"Rows where x > 12m:")
high_x

---
## 4. Data Manipulation

In [ ]:
# Numerical differentiation (velocity)
df['vx'] = np.gradient(df.x,df.time)
df['vy'] = np.gradient(df.y,df.time)
df['vz'] = np.gradient(df.z,df.time)

In [ ]:
# Create a new column: velocity magnitude
df['v_mag'] = np.sqrt(
    df['vx']**2 + 
    df['vy']**2 + 
    df['vz']**2
)

print("Added velocity_magnitude column:")
df[['vx', 'vy', 'vz', 'v_mag']].head()

---
## 5. NumPy Operations

Pandas and NumPy work seamlessly together.

In [ ]:
# Convert DataFrame columns to NumPy arrays
time_array = df['time'].values
temp_array = df['x'].values

print(f"Type: {type(time_array)}")
print(f"Array shape: {time_array.shape}")

In [ ]:
# NumPy statistical operations
print(f"Mean x: {np.mean(temp_array):.2f} m")
print(f"Std x: {np.std(temp_array):.2f} m")
print(f"Max x: {np.max(temp_array):.2f} m")
print(f"Min x: {np.min(temp_array):.2f} m")

In [ ]:
# Numerical integration (total displacement in the x-direction)
# Using trapezoidal rule
total_displacement_x = np.trapezoid(df['vx'], df['time'])
print(f"Total X displacement: {total_displacement_x:.2f} m")

---
## 6. Visualization with Matplotlib

In [ ]:
# Simple time series plot
fig, ax = plt.subplots(figsize=(8, 6))

# ax.plot(df['x'], df['y'], 'b-', linewidth=2)
# ax.plot(df.x, df.y, 'b-', linewidth=2)

# ax.set_xlabel('x [m]', fontsize=12)
# ax.set_ylabel('y [m]', fontsize=12)
# ax.set_title('Trajectory in xy-plane', fontsize=14)
# ax.grid(True, alpha=0.3)

# plt.axis('equal')
# plt.tight_layout()
# plt.show()

In [ ]:
fs = 14
fs2 = fs+2
fs3 = fs+4
# Setting plotting parameters manually
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rc('xtick',labelsize=fs)
plt.rc('ytick',labelsize=fs)


# Restore matplotlib defaults if desired
# plt.rcdefaults()
# Change plot style
# plt.style.use('seaborn-v0_8-whitegrid')
# plt.rcParams["font.family"] = "Times New Roman"  # overrides style
# plt.rcParams['mathtext.fontset'] = 'stix'

In [ ]:
# Simple time series plot
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(df['x'], df['z'], 'g', linewidth=2)
ax.set_xlabel('x [m]', fontsize=fs2)
ax.set_ylabel('z [m]', fontsize=fs2)
ax.set_title('Trajectory in xz-plane', fontsize=fs3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------
# Static subplots for planes
# -------------------------------
fig, axs = plt.subplots(1, 2, figsize=(12,5))

# x-y plane
axs[0].plot(df['time'], df['x'], color='blue', label = 'x')
axs[0].plot(df['time'], df['y'], color='green', label = 'y')
axs[0].plot(df['time'], df['z'], color='red', label = 'z')

axs[0].legend(fontsize = fs)
axs[0].set_xlabel('Time [s]', fontsize=fs2)
axs[0].set_ylabel('Position', fontsize=fs3)
axs[0].set_title('Position vs. Time', fontsize=fs3)

axs[0].spines['right'].set_visible(False)
axs[0].spines['top'].set_visible(False)
axs[0].yaxis.set_ticks_position('left')
axs[0].xaxis.set_ticks_position('bottom')

# x-z plane
axs[1].plot(df['time'], df['vx'], color='blue', label = r'$v_x$')
axs[1].plot(df['time'], df['vy'], color='green', label = r'$v_y$')
axs[1].plot(df['time'], df['vz'], color='red', label = r'$v_z$')

axs[1].legend(fontsize = fs)
axs[1].set_xlabel('Time [s]', fontsize=fs2)
axs[1].set_ylabel('Velocity', fontsize=fs3)
axs[1].set_title('Velocity vs. Time', fontsize=fs3)

axs[1].spines['right'].set_visible(False)
axs[1].spines['top'].set_visible(False)
axs[1].yaxis.set_ticks_position('left')

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------
# Static subplots for planes
# -------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# scatter = ax.scatter(
#     df['x'], 
#     df['y'], 
#     c=df['v_mag'], 
#     cmap='viridis',
#     s=50,
#     alpha=0.7
# )

# cbar = plt.colorbar(scatter)
# cbar.set_label('Velocity Magnitude [m/s]', fontsize=fs2, labelpad = 15)

# ax.set_xlabel('x [m]', fontsize=fs2)
# ax.set_ylabel('y [m]', fontsize=fs3)
# ax.set_title(r'Trajectory in xy-Plane Colored by $|\vec{v}|$', fontsize=fs3)

# ax.spines['right'].set_visible(False)
# ax.spines['top'].set_visible(False)
# ax.yaxis.set_ticks_position('left')
# ax.xaxis.set_ticks_position('bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Publication-quality figure
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(df['x'], df['y'], 'b-', linewidth=2, label='y')
#ax.fill_between(df['x'], df['y'], alpha=0.3)

ax.set_xlabel('x [m]', fontsize=fs2)
ax.set_ylabel('y [m]', fontsize=fs2)
ax.set_title('Trajectory in xy-Plane', fontsize=fs3, fontweight='bold')
#ax.legend(fontsize=fs, loc = 'upper left')

# Add annotations
# max_idx = df['y'].idxmax()
# ax.annotate(
#     f'Max: {df.loc[max_idx, "y"]:.1f} m/s',
#     xy=(df.loc[max_idx, 'x'], df.loc[max_idx, 'y']),
#     xytext=(10, 13),
#     fontsize=fs-2,
#     arrowprops=dict(arrowstyle='->', color='red')
# )

ax.grid(True, alpha=0.3)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.yaxis.set_ticks_position('left')
ax.xaxis.set_ticks_position('bottom')

plt.tight_layout()

# Save the figure
# fig.savefig('temperature_plot.png', dpi=300, bbox_inches='tight')

plt.show()

---
## 7. Visualization with vtkplotlib

In [ ]:
# -------------------------------
# Extract positions
# -------------------------------
# x = df['x'].values
# y = df['y'].values
# z = df['z'].values
# points = np.column_stack((x, y, z))

# x_plane = np.linspace(min(x)-1, max(x)+1, 10)
# z_plane = np.linspace(min(z)-1, max(z)+1, 10)
# X_plane, Z_plane = np.meshgrid(x_plane, z_plane)
# Y_plane = np.zeros_like(X_plane)  # y = 0 plane

# -------------------------------
# Plot trajectory and axes
# -------------------------------
#vpl.figure()

# Draw XZ plane (gree, semi-transparent)
#vpl.surface(X_plane, Y_plane, Z_plane, color='green', opacity=0.5)

# Plot trajectory with colormap of speed
# traj = vpl.plot(points, color=df['v_mag'].values, cmap='viridis', line_width=3)
# traj.scalar_range = [0, 80] # Set color limits
# vpl.scalar_bar(traj, title='Speed (m/s)')


# Draw XYZ axes as arrows
# vpl.arrow(start=[0,0,0], end=[3,0,0], color='red', width_scale=1.0)    # X
# vpl.arrow(start=[0,0,0], end=[0,3,0], color='green', width_scale=1.0)  # Y
# vpl.arrow(start=[0,0,0], end=[0,0,3], color='blue', width_scale=1.0)   # Z


# Show figure
#vpl.show()

---
## 8. Saving Processed Data

In [ ]:
# Save to CSV
# df.to_csv('data/processed_data.csv', index=False)
# print("Data saved to processed_data.csv")

In [ ]:
# Save to Excel (requires openpyxl)
# df.to_excel('data/processed_data.xlsx', index=False)
# print("Data saved to processed_data.xlsx")

---
## Part 2: LES Cylinder CFD Data (Complex Visualization)

In this second part, we switch from a small experimental-style CSV to a **large-eddy simulation (LES) cylinder flow** case.

We will:
- Load a precomputed LES grid (`cylinder.sp.x`) and a few flow snapshots (`sol-*.q`) using `lptlib`.
- Extract a 2D mid-plane and compute the streamwise velocity component \(u\).
- Visualize \(u(x, y)\) with a contour plot.
- (Optionally) animate how the velocity field evolves over time.

> **Note:** This section assumes you have a small subset of LES files in `hour2_scientific/data/cylinder/` (see that folder's `README.md`) and that `lptlib` is installed in your environment.

In [ ]:
# Imports for LES cylinder data (Part 2)
try:
    # lptlib provides GridIO and FlowIO utilities for structured CFD data
    from lptlib.io.plot3dio import GridIO, FlowIO


    print("Libraries loaded successfully!")
    print("lptlib imported successfully.")
except ImportError:
    print("lptlib is not installed. To run the LES cylinder section, install it with e.g. `pip install lptlib` or your local lptlib source.")


In [ ]:
# Import helper functions and animation utilities from modules
DATA_DIR = os.path.join("hour2_scientific", "data", "cylinder") if os.getcwd().endswith("dc-qc-sci-python") else os.path.join("data", "cylinder")

# Import helper functions and animation utilities
from modules.animation import animate_u_contours
from modules.helpers import prepare_midplane_u

print("Modules imported successfully!")


In [ ]:
# Load grid and a single LES snapshot
# Grid and one example flow file (small subset shipped in hour2_scientific/data/cylinder)
grid_path = os.path.join(DATA_DIR, "cylinder.sp.x")
example_flow_path = os.path.join(DATA_DIR, "sol-0000010.q")

print("Grid path:", grid_path)
print("Example flow file:", example_flow_path)

if not os.path.exists(grid_path):
    raise FileNotFoundError(f"Grid file not found: {grid_path}.\nCheck that you've copied 'cylinder.sp.x' into data/cylinder/.")

if not os.path.exists(example_flow_path):
    raise FileNotFoundError(
        f"Example flow file not found: {example_flow_path}.\n"
        "Copy a few 'sol-*.q' files into data/cylinder/ as described in data/cylinder/README.md."
    )

# Read grid and flow
grid = GridIO(grid_path)
grid.read_grid()
flow = FlowIO(example_flow_path)
flow.read_flow(data_type="f4")

In [ ]:
# Static contour plot of u-velocity on the mid-plane

fig, ax = plt.subplots(figsize=(8, 3))

blocks = prepare_midplane_u(grid, flow)
contours = []
for X, Y, U in blocks:
    cs = ax.contourf(X, Y, U, levels=40, cmap="RdBu_r")
    contours.append(cs)

cbar = fig.colorbar(contours[-1], ax=ax)
cbar.set_label("u-velocity")

# Match the zoom/axes from the original scripts (same as animation)
ax.set_xlim(-2, 12)
ax.set_ylim(-2, 2)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("LES cylinder: u-velocity mid-plane (single snapshot)")

plt.tight_layout()
plt.show()

### Interactive Animation with Controls

The animation below will open in a **separate interactive window** with play/pause controls, step forward/backward buttons, and a slider to navigate through frames.

> **Note:** For the interactive window to appear, you may need to install a GUI backend:
> - Windows/Linux: Usually works with default `tkinter` (comes with Python)
> - If needed: `pip install PyQt5` for better performance
> - If the window doesn't appear, the animation will display inline instead

In [ ]:
# Simple LES u-velocity animation over a few timesteps (interactive pop-out window)
%matplotlib qt
anim = animate_u_contours(grid_path=grid_path, example_flow_path=example_flow_path,
                       interval_ms=80, levels=40, save_path=None)

# Note: The animation window should open automatically. Keep a reference to 'anim' 
# to prevent it from being garbage collected.

# Optional: save to mp4 if ffmpeg is available (commented out by default)
# anim.save("les_cylinder_u_animation.mp4", dpi=150, writer="ffmpeg")


In [ ]:
# change back to inline plotting by uncommenting the following line
%matplotlib inline

---
## Part 3: Spectral Proper Orthogonal Decomposition


In [ ]:
# -----------------------
# Grid
# -----------------------
nx, ny = 100, 100
Length = 10.0

x = np.linspace(-Length/2, Length/2, nx)
y = np.linspace(-Length/2, Length/2, ny)
X, Y = np.meshgrid(x, y, indexing="xy")
R = np.sqrt(X**2 + Y**2)

# -----------------------
# Time parameters
# -----------------------
fps = 20000
dt = 1 / fps
nt_full = 1024*5      # full field length
nt_anim = 100       # frames to animate

t_full = np.arange(nt_full) * dt
t_anim = t_full[:nt_anim]

# -----------------------
# Wave parameters
# -----------------------
# Circular wave
freq1 = 500
omega1 = 2 * np.pi * freq1
k1 = 8.0
A1 = 1.0

# Diagonal traveling wave
freq2 = 1200
omega2 = 2 * np.pi * freq2
A2 = 0.4

k2 = 3.0
kx = k2 / np.sqrt(2)
ky = k2 / np.sqrt(2)

# Noise parameters
noise_amp = 0.075  # small amplitude
np.random.seed(42)  # for reproducibility

# -----------------------
# Precompute full wavefield with noise
# -----------------------
field_full = np.empty((nt_full, ny, nx), dtype=np.float32)

for i, ti in enumerate(t_full):
    wave = (
        A1 * np.sin(k1 * R - omega1 * ti) +
        A2 * np.sin(kx * X + ky * Y - omega2 * ti)
    )
    noise = noise_amp * np.random.randn(ny, nx)
    field_full[i] = wave + noise

print("Full field shape:", field_full.shape)

In [ ]:
# -----------------------
# Figure setup
# -----------------------
fig, ax = plt.subplots(dpi=150)
vmax = 1.5
norm = plt.Normalize(vmin=-vmax, vmax=vmax)

im = ax.imshow(
    field_full[0],
    extent=[x.min(), x.max(), y.min(), y.max()],
    origin="lower",
    cmap=cm.viridis,
    norm=norm
)

plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(im)
ax.set_title("")

# -----------------------
# Animation function
# -----------------------
def animate(i):
    im.set_array(field_full[i])
    ax.set_title(
        f"Circular: {freq1:.1f} Hz | Diagonal: {freq2:.1f} Hz\n"
        f"Time: {1000*t_full[i]:.2f} ms | Cycles (circ): {t_full[i]*freq1:.2f}"
    )
    return im,

# -----------------------
# Create animation (first 300 frames)
# -----------------------
anim = FuncAnimation(
    fig,
    animate,
    frames=nt_anim,
    interval=50,  # slower playback
    blit=True
)

# anim.save("combined_waves_noise.mp4", writer="ffmpeg", fps=20)
plt.close()
#HTML(anim.to_html5_video()) # Only works if ffmpeg.exe is installed
HTML(anim.to_jshtml())



In [ ]:
# Perform the SPOD

save_dir = './data/spod_data'

# Check if it exists, and create it if it doesn't
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Directory created: {save_dir}")
else:
    print(f"Directory already exists: {save_dir}")

Q = field_full.reshape(nt_full,-1) # Flatten array to be 1D

nfft = 512 # Changing this effects the SPOD
spod.spod(Q,dt,save_dir,weight='default',window='default',method='fast',nDFT=nfft)


In [ ]:
#Load SPOD
load_file = os.path.join(save_dir,'SPOD_LPf.h5')
SPOD_LPf  = h5py.File(load_file,'r') # load data from h5 format
L = SPOD_LPf['L'][:,:]    # modal energy E(f, M)
P = SPOD_LPf['P'][:,:,:]  # mode shape
f = SPOD_LPf['f'][:]      # frequency
SPOD_LPf.close()
os.remove(load_file) # Delete file since it is large

In [ ]:
# PLot the mode spectra
fig = spod.plot_spectrum(f,L,hl_idx=5)

In [ ]:
# Find spectral peaks

mode = 0 # Mode number of interest
peaks, properties = find_peaks(L[:,mode])

# Get the peak heights
peak_heights = properties.get('peak_heights', L[:,mode][peaks])

# Sort peaks by height descending
sorted_indices = np.argsort(peak_heights)[::-1]

# Get the indices of the two highest peaks
top2_peaks = peaks[sorted_indices[:2]]

print("Indices of top 2 peaks:", top2_peaks)
print(f"Mode {mode+1} frequencies:", f[top2_peaks])


### Visualize SPOD Modes


In [ ]:
# Prepare time array for one block
time_per_block = nfft * dt

t = np.linspace(0, 0.25*time_per_block, nt_anim)

# frequency, mode selector
freq=top2_peaks[0]

# Create figure and axes, but don't plot anything initially
fig, ax = plt.subplots(dpi=150)
image_data = np.real((P[freq,:,mode] * np.exp(-1j * 2 * np.pi * f[freq] * 0))).reshape(nx, ny)

im = ax.imshow(image_data,extent=[x.min(), x.max(), y.min(), y.max()],origin="lower",cmap = 'viridis')
plt.xlabel('x')
plt.ylabel('y')
plt.colorbar(im)
ax.set_title("")

# Animation function
def animate(i):
    # Compute the mode data at this time step
    image_data = np.real((P[freq, :, mode] * np.exp(1j * 2 * np.pi * f[freq] * t[i]))).reshape(nx, ny)
    
  
    # Removed masking:
    # rgba_image[mask, :] = [0, 0, 0, 1]
    
    # Update the image for animation
    im.set_array(image_data)
    
    # Update title with frequency, mode, and time info
    ax.set_title(
        f"f={f[freq]} Hz, Mode={mode}\n"
        f"Time: {1000*t[i]:.2f} ms\n"
        f"Cycles: {t[i]/(1/f[freq]):.1f}"
    )
    
    return im,


# Create animation
anim = FuncAnimation(fig, animate, frames=len(t), interval=30, blit=True)
#anim.save(os.path.join(save_dir,f'SPOD_f{int(f[freq])}m{mode+1}.mp4'), writer='ffmpeg', fps=20)
plt.close()
#HTML(anim.to_html5_video()) # Only works if ffmpeg.exe is installed
HTML(anim.to_jshtml())


In [ ]:
# Prepare time array for one block
time_per_block = nfft * dt

t = np.linspace(0, 0.25*time_per_block, nt_anim)

# frequency, mode selector
freq=top2_peaks[1]

# Create figure and axes, but don't plot anything initially
fig, ax = plt.subplots(dpi=150)
image_data = np.real((P[freq,:,mode] * np.exp(-1j * 2 * np.pi * f[freq] * 0))).reshape(ny, nx)

im = ax.imshow(image_data,extent=[x.min(), x.max(), y.min(), y.max()],origin="lower",cmap = 'viridis')
plt.xlabel('x')
plt.ylabel('y')
plt.colorbar(im)
ax.set_title("")

# Animation function
def animate(i):
    # Compute the mode data at this time step
    image_data = np.real((P[freq, :, mode] * np.exp(1j * 2 * np.pi * f[freq] * t[i]))).reshape(nx, ny)
    
  
    # Removed masking:
    # rgba_image[mask, :] = [0, 0, 0, 1]
    
    # Update the image for animation
    im.set_array(image_data)
    
    # Update title with frequency, mode, and time info
    ax.set_title(
        f"f={f[freq]} Hz, Mode={mode}\n"
        f"Time: {1000*t[i]:.2f} ms\n"
        f"Cycles: {t[i]/(1/f[freq]):.1f}"
    )
    
    return im,


# Create animation
anim = FuncAnimation(fig, animate, frames=len(t), interval=30, blit=True)
#anim.save(os.path.join(save_dir,f'SPOD_f{int(f[freq])}m{mode+1}.mp4'), writer='ffmpeg', fps=20)
plt.close()
#HTML(anim.to_html5_video()) # Only works if ffmpeg.exe is installed
HTML(anim.to_jshtml())

### SPOD on Cylinder LES

The following two cells can be used to download and extract a larger portion of the LES for modal analysis. Note that some cropping and data masking may be required. 


In [ ]:
# Download LES cylinder data for the POD / animation demo
from modules.helpers import download_cylinder_les_data

# uncomment to download the whole cynlinder data
#dataset_dir = download_cylinder_les_data()

In [ ]:
# Build the stacked cylinder mid-plane u-velocity array
# Final shape: (t, x, y)

from modules.helpers import stack_midplane_u_over_time

#U = stack_midplane_u_over_time(grid, example_flow_path, block_idx=0, data_type="f4")
#print(f"Final stacked U shape (t, x, y): {U.shape}")


---
## Summary

You've learned how to:
- Load data from CSV files using Pandas
- Explore datasets (shape, dtypes, statistics, missing values)
- Access and filter data
- Create new columns and perform calculations
- Use NumPy for numerical operations
- Create various types of plots with Matplotlib
- 3D Visualization with Vtkplotlib
- Visualization of timeseries LES data
- Spectral Proper Orthogonal Decomposition and visualization

**Next:** Move to Hour 3 to learn about Deep Neural Networks with PyTorch!